In [1]:
from sqlalchemy import create_engine, inspect, text
import pandas as pd

host = "localhost"
port = "3306"
username = "ndadmin"
password = "ndADMIN%402025"
schema = "deidentified_oct"
# schema = "mobiledoc_staging_1d_deidentified"
# schema = "mobiledoc_staging_deidentified"
folder = "unique_key_fix"

# Connect to Database
engine = create_engine(f"mysql+pymysql://{username}:{password}@{host}:{port}/{schema}")

In [2]:
def add_unique_constraint(conn, table_name):
    # Check if column exists
    column_exists = conn.execute(
        text("""
            SELECT 1
            FROM information_schema.COLUMNS
            WHERE TABLE_SCHEMA = :schema
              AND TABLE_NAME = :table
              AND COLUMN_NAME = 'nd_auto_increment_id'
            LIMIT 1
        """),
        {"schema": schema, "table": table_name},
    ).fetchone()

    if not column_exists:
        return

    # Check if UNIQUE already exists
    has_unique = conn.execute(
        text("""
            SELECT 1
            FROM information_schema.STATISTICS
            WHERE TABLE_SCHEMA = :schema
              AND TABLE_NAME = :table
              AND COLUMN_NAME = 'nd_auto_increment_id'
              AND NON_UNIQUE = 0
            LIMIT 1
        """),
        {"schema": schema, "table": table_name},
    ).fetchone()

    if has_unique:
        print(f"✅ {table_name}: UNIQUE already exists")
        return

    # Add UNIQUE index
    print(f"🔧 {table_name}: adding UNIQUE constraint")

    conn.execute(text(f"""
        ALTER TABLE `{schema}`.`{table_name}`
        ADD UNIQUE INDEX uniq_nd_auto_increment_id (nd_auto_increment_id)
    """))

def deduplicate_keep_one(conn, table):
    print(f"🧹 Deduplicating {table}")

    sql = text(f"""
        DELETE t
        FROM `{schema}`.`{table}` t
        JOIN (
            SELECT
                nd_auto_increment_id,
                nd_created_at,
                ROW_NUMBER() OVER (
                    PARTITION BY nd_auto_increment_id
                    ORDER BY nd_created_at
                ) AS rn
            FROM `{schema}`.`{table}`
        ) d
        ON t.nd_auto_increment_id = d.nd_auto_increment_id
        AND t.nd_created_at       = d.nd_created_at
        WHERE d.rn > 1;
    """)

    conn.execute(text("set sql_safe_updates = 0;"))
    result = conn.execute(sql)
    conn.execute(text("set sql_safe_updates = 1;"))
    print(f"🗑️ {table}: removed {result.rowcount} duplicate rows")

def main():
    with engine.begin() as conn:
        tables = conn.execute(
            text("""
                SELECT TABLE_NAME
                FROM information_schema.TABLES
                WHERE TABLE_SCHEMA = :schema
            """),
            {"schema": schema},
        ).scalars().all()
        # tables = ['referraldetail', 'referralEx', 'referralstatus', 'visitcodes']

        print(f"Found {len(tables)} tables")

        for table in tables:
            try:
                add_unique_constraint(conn, table)
            except:
                deduplicate_keep_one(conn, table)
                add_unique_constraint(conn, table)

    print("🎉 UNIQUE constraint added to all applicable tables")

In [3]:
main()

Found 58 tables
✅ allergies: UNIQUE already exists
✅ annualnotes: UNIQUE already exists
✅ assessment_notes_history: UNIQUE already exists
✅ billingdata: UNIQUE already exists
✅ cpt_validcodes: UNIQUE already exists
✅ cptcode_base: UNIQUE already exists
✅ doctors: UNIQUE already exists
✅ edi_dfr_info: UNIQUE already exists
✅ edi_inv_cpt: UNIQUE already exists
✅ edi_inv_diagnosis: UNIQUE already exists
✅ edi_invoice: UNIQUE already exists
✅ enc: UNIQUE already exists
✅ encaddendums: UNIQUE already exists
✅ encounterData: UNIQUE already exists
✅ encounters: UNIQUE already exists
✅ family: UNIQUE already exists
✅ hcpcscode_base: UNIQUE already exists
✅ hl7labdatadetail: UNIQUE already exists
✅ hl7labnotes: UNIQUE already exists
✅ hpi: UNIQUE already exists
✅ icd10cm_desc: UNIQUE already exists
✅ icd_9: UNIQUE already exists
✅ immunizations: UNIQUE already exists
✅ inpatientvisit: UNIQUE already exists
✅ InteractionNotes: UNIQUE already exists
✅ items: UNIQUE already exists
✅ labdata: UNIQU

In [4]:
# annualnotes, billingdata, edi_inv_cpt